# 1. Surface Topology Graph — Voronoi Representation

This notebook demonstrates the **Voronoi-based surface graph representation** for catalytic surfaces.

**What you'll do:**
- Build a Cu(111) slab from scratch
- Compute Voronoi topology graph (nodes = atoms, edges = bonds)
- Extract GNN-ready node features (6 dimensions)
- Classify adsorption sites (top, bridge, hollow_fcc, hollow_hcp)
- Visualise the surface graph and site symmetry scores

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1.1 Build a Cu(111) slab
We create a simple 3×3×2 Cu(111) slab with 15 Å vacuum.

In [ ]:
# Build slab using ASE
from ase.build import fcc111
from ase.visualize.plot import plot_atoms

slab = fcc111('Cu', size=(3, 3, 3), vacuum=15.0, a=3.615)
print(f'Atoms: {len(slab)}, Cell: {slab.cell.lengths()}')
print(f'Elements: {set(slab.get_chemical_symbols())}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_atoms(slab, axes[0], rotation='0x,0y,0z')  # top view
axes[0].set_title('Top view')
plot_atoms(slab, axes[1], rotation='90x,0y,0z')  # side view
axes[1].set_title('Side view')
plt.tight_layout()
plt.show()

## 1.2 Build the Voronoi Topology Graph
The `SurfaceTopologyGraph` uses Voronoi tessellation with periodic images to identify:
- **Nodes**: atoms with features [Z, layer, CN, Voronoi_vol, surface_dist, angle_var]
- **Edges**: Voronoi-neighbour bonds with [length, angle_with_normal, face_area]

In [ ]:
from science.representations.surface_graph import SurfaceTopologyGraph

stg = SurfaceTopologyGraph(
    positions=slab.get_positions(),
    elements=slab.get_chemical_symbols(),
    cell=slab.get_cell()[:],
)
stg.build()
print(stg.summary())
print(f'\nNodes: {len(stg.nodes)}')
print(f'Edges: {len(stg.edges)}')
print(f'\nFirst node: element={stg.nodes[0].element}, layer={stg.nodes[0].layer}, CN={stg.nodes[0].coordination}')

## 1.3 Node Feature Matrix (GNN input)
6 features per atom, normalised to O(1) range — ready for PyTorch Geometric.

In [ ]:
X = stg.node_feature_matrix()
print(f'Feature matrix shape: {X.shape}  (N_atoms × 6)')
print(f'Feature names: [Z/100, layer, CN/12, Voronoi_vol/20, surface_dist/5, angle_var]')
print(f'\nFeature statistics:')
for i, name in enumerate(['Z/100', 'layer', 'CN/12', 'vol/20', 'dist/5', 'angle_var']):
    print(f'  {name:12s}  min={X[:,i].min():.3f}  max={X[:,i].max():.3f}  mean={X[:,i].mean():.3f}')

In [ ]:
# Edge index for PyTorch Geometric
ei, ea = stg.edge_index_and_attr()
print(f'edge_index shape: {ei.shape}  (2 × 2E, bidirectional)')
print(f'edge_attr shape:  {ea.shape}  (2E × 3: [length/5, angle/pi, area/10])')
print(f'\nMean bond length: {ea[:,0].mean()*5:.2f} Å')
print(f'Mean Voronoi face area: {ea[:,2].mean()*10:.2f} Å²')

## 1.4 Adsorption Site Classification
Sites are classified geometrically via Delaunay triangulation + subsurface atom detection:
- **top**: above each surface atom
- **bridge**: midpoint of surface edges
- **hollow_fcc**: triangle centroid, no subsurface atom below
- **hollow_hcp**: triangle centroid, subsurface atom below

In [ ]:
sites = stg.classify_adsorption_sites(ads_height=1.8)
print(f'Found {len(sites)} adsorption sites:\n')

from collections import Counter
site_counts = Counter(s.site_type for s in sites)
for stype, count in sorted(site_counts.items()):
    avg_sym = np.mean([s.symmetry_rank for s in sites if s.site_type == stype])
    print(f'  {stype:12s}  count={count}  avg_symmetry={avg_sym:.3f}')

In [ ]:
# Visualise sites on the surface
fig, ax = plt.subplots(figsize=(8, 7))
colors = {'top': 'red', 'bridge': 'blue', 'hollow_fcc': 'green', 'hollow_hcp': 'orange'}

# Plot atoms
surface_atoms = [n for n in stg.nodes if n.layer == 0]
for a in surface_atoms:
    ax.plot(a.position[0], a.position[1], 'ko', markersize=12, alpha=0.3)

# Plot sites
for s in sites:
    ax.plot(s.position[0], s.position[1], 'o',
            color=colors.get(s.site_type, 'gray'),
            markersize=8 * (0.5 + s.symmetry_rank),
            alpha=0.7, label=s.site_type)

# Legend (deduplicate)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right')
ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)')
ax.set_title('Adsorption sites on Cu(111) — size ∝ symmetry score')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 1.5 Export to NetworkX
The graph can be exported for further analysis or GNN training.

In [ ]:
G = stg.to_networkx()
print(f'NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Node 0 features: {G.nodes[0]["features"]}')
print(f'Node 0 element: {G.nodes[0]["element"]}, layer: {G.nodes[0]["layer"]}')